### Notebook 1 - Extracción y preparación de datos de mercado

Primera fase del pipeline técnico del TFM: descarga de precios históricos de los ETFs seleccionados vía `yfinance`, limpieza de la ventana temporal común entre activos, construcción del `dataset_mercado` (precios, retornos, volatilidades, drawdowns) y generación del `catalogo_carteras` (286 combinaciones de pesos por activo, clasificadas por perfil de riesgo), y del `dataset_colateral` (evolución del valor del colateral por cartera).

**Salida:** `data/dataset_mercado.parquet`, `data/catalogo_carteras.parquet`, `data/dataset_colateral.parquet`.

**Nota de ejecución:** este notebook requiere acceso a internet para descargar precios vía `yfinance`.

In [1]:
# ============================================================
# 0. Librerías y configuración
# ============================================================

import pandas as pd
import numpy as np
import yfinance as yf
from itertools import product

NOMBRE_ACTIVOS = ["XEON.DE", "SHY", "AGGU.L", "SPY"]

START_DATE = "2000-01-01"
END_DATE = "2026-06-30"
INTERVAL = "1d"

TER = {
    "XEON.DE": 0.0010,
    "SHY": 0.0015,
    "AGGU.L": 0.0010,
    "SPY": 0.0009
}

SPREAD = 0.025
EURIBOR_3M_ASSUMED = 0.035

In [2]:
# ============================================================
# 1. Descargar precios
# ============================================================

def descargar_precios(tickers, start_date, interval, end_date):
    df_raw = yf.download(
        tickers=tickers,
        start=start_date,
        end=end_date,
        interval=interval,
        auto_adjust=False,
        actions=True,
        repair=True,
        group_by="ticker",
        progress=True,
        threads=True
    )
    return df_raw


def extraer_adj_close(df_raw, tickers):
    precios = pd.DataFrame()

    for ticker in tickers:
        precios[ticker] = df_raw[ticker]["Adj Close"]

    precios = precios.dropna(how="all").sort_index()
    precios.index.name = "date"

    return precios

In [3]:
# ============================================================
# 2. Ventana común
# ============================================================

def limpiar_ventana_comun(precios):
    resumen = []

    for col in precios.columns:
        serie = precios[col].dropna()
        resumen.append({
            "etf": col,
            "fecha_min": serie.index.min(),
            "fecha_max": serie.index.max(),
            "registros_validos": serie.shape[0]
        })

    df_resumen = pd.DataFrame(resumen)

    fecha_inicio = df_resumen["fecha_min"].max()
    fecha_fin = df_resumen["fecha_max"].min()

    precios_limpios = precios.loc[fecha_inicio:fecha_fin].dropna()

    return precios_limpios, df_resumen, fecha_inicio, fecha_fin

In [4]:
# ============================================================
# 3. Dataset mercado
# ============================================================

def calcular_drawdown(precios, window=63):
    max_rolling = precios.rolling(
        window=window,
        min_periods=window
    ).max()

    drawdown = precios / max_rolling - 1

    return drawdown


def calcular_corr_media_rolling(retornos, window=63):
    corr_medias = []

    for i in range(len(retornos)):
        if i < window:
            corr_medias.append(np.nan)
        else:
            corr = retornos.iloc[i-window:i].corr()
            upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
            corr_medias.append(upper.stack().mean())

    return pd.Series(corr_medias, index=retornos.index)


def construir_dataset_mercado(precios):
    retornos = precios.pct_change()
    retornos_log = np.log(precios / precios.shift(1))

    dataset = pd.DataFrame(index=precios.index)

    for ticker in precios.columns:
        dataset[f"price_{ticker}"] = precios[ticker]

        dataset[f"ret_1d_{ticker}"] = retornos[ticker]
        dataset[f"ret_log_1d_{ticker}"] = retornos_log[ticker]

        dataset[f"ret_21d_{ticker}"] = precios[ticker].pct_change(21)
        dataset[f"ret_63d_{ticker}"] = precios[ticker].pct_change(63)
        dataset[f"ret_252d_{ticker}"] = precios[ticker].pct_change(252)

        dataset[f"vol_21d_{ticker}"] = (
            retornos[ticker]
            .rolling(window=21, min_periods=21)
            .std() * np.sqrt(252)
        )

        dataset[f"vol_63d_{ticker}"] = (
            retornos[ticker]
            .rolling(window=63, min_periods=63)
            .std() * np.sqrt(252)
        )

        dataset[f"vol_252d_{ticker}"] = (
            retornos[ticker]
            .rolling(window=252, min_periods=252)
            .std() * np.sqrt(252)
        )

    drawdown_63d = calcular_drawdown(precios, window=63)

    for ticker in precios.columns:
        dataset[f"drawdown_63d_{ticker}"] = drawdown_63d[ticker]

    dataset["corr_media_63d"] = calcular_corr_media_rolling(
        retornos.dropna(),
        window=63
    )

    dataset["euribor_3m"] = EURIBOR_3M_ASSUMED
    dataset["spread"] = SPREAD
    dataset["coste_financiacion"] = dataset["euribor_3m"] + dataset["spread"]

    dataset = dataset.reset_index()

    return dataset, retornos.dropna(), retornos_log.dropna()

In [5]:
# ============================================================
# 4. Catálogo de carteras
# ============================================================

def clasificar_perfil(pesos, tickers):
    w = dict(zip(tickers, pesos))

    peso_rv = w.get("SPY", 0)
    peso_monetario = w.get("XEON.DE", 0)
    peso_rf = w.get("SHY", 0) + w.get("AGGU.L", 0)

    if peso_rv <= 0.20 and peso_monetario + peso_rf >= 0.80:
        return "conservador"
    elif peso_rv <= 0.50:
        return "equilibrado"
    else:
        return "dinamico"


def generar_catalogo_carteras(tickers, step=0.10):
    valores = np.round(np.arange(0, 1 + step, step), 4)

    combinaciones = []

    for pesos in product(valores, repeat=len(tickers)):
        if np.isclose(sum(pesos), 1.0):
            pesos = np.array(pesos, dtype=float)

            row = {
                "id_cartera": len(combinaciones) + 1,
                "perfil": clasificar_perfil(pesos, tickers),
                "hhi_concentracion": np.sum(pesos ** 2)
            }

            ter_cartera = 0

            for ticker, peso in zip(tickers, pesos):
                row[f"w_{ticker}"] = peso
                ter_cartera += peso * TER[ticker]

            row["ter_cartera"] = ter_cartera

            combinaciones.append(row)

    catalogo = pd.DataFrame(combinaciones)

    return catalogo

In [6]:
# ============================================================
# 5. Ejecución
# ============================================================

df_raw = descargar_precios(NOMBRE_ACTIVOS, START_DATE, INTERVAL, END_DATE)
print("Datos descargados.")

precios = extraer_adj_close(df_raw, NOMBRE_ACTIVOS)
print("Precios extraídos.")

precios_limpios, df_resumen, fecha_inicio, fecha_fin = limpiar_ventana_comun(precios)
print("Ventana común calculada.")

dataset_mercado, retornos, retornos_log = construir_dataset_mercado(precios_limpios)
print("Dataset de mercado construido.")

catalogo_carteras = generar_catalogo_carteras(NOMBRE_ACTIVOS, step=0.10)
print("Catálogo de carteras construido.")
catalogo_carteras

[*********************100%***********************]  4 of 4 completed


Datos descargados.
Precios extraídos.
Ventana común calculada.
Dataset de mercado construido.
Catálogo de carteras construido.


,id_cartera,perfil,hhi_concentracion,w_XEON.DE,w_SHY,w_AGGU.L,w_SPY,ter_cartera
0,1,dinamico,1.00,0.0,0.0,0.0,1.0,0.00090
1,2,dinamico,0.82,0.0,0.0,0.1,0.9,0.00091
2,3,dinamico,0.68,0.0,0.0,0.2,0.8,0.00092
3,4,dinamico,0.58,0.0,0.0,0.3,0.7,0.00093
4,5,dinamico,0.52,0.0,0.0,0.4,0.6,0.00094
...,...,...,...,...,...,...,...,...
281,282,conservador,0.68,0.8,0.2,0.0,0.0,0.00110
282,283,conservador,0.82,0.9,0.0,0.0,0.1,0.00099
283,284,conservador,0.82,0.9,0.0,0.1,0.0,0.00100
284,285,conservador,0.82,0.9,0.1,0.0,0.0,0.00105


In [7]:
# ============================================================
# 6. Validaciones
# ============================================================

print("Fecha inicio común:", fecha_inicio)
print("Fecha fin común:", fecha_fin)

print("\nResumen cobertura:")
display(df_resumen)

print("\nShape precios limpios:", precios_limpios.shape)
print("Shape dataset mercado:", dataset_mercado.shape)
print("Shape catálogo carteras:", catalogo_carteras.shape)

print("\nDistribución perfiles:")
display(catalogo_carteras["perfil"].value_counts())

print("\nNulos dataset mercado:")
display(dataset_mercado.isnull().sum())

display(dataset_mercado.head())
display(catalogo_carteras.head())

Fecha inicio común: 2017-11-21 00:00:00
Fecha fin común: 2026-06-29 00:00:00

Resumen cobertura:


,etf,fecha_min,fecha_max,registros_validos
0,XEON.DE,2008-01-02,2026-06-29,4696
1,SHY,2002-07-30,2026-06-29,6017
2,AGGU.L,2017-11-21,2026-06-29,2171
3,SPY,2000-01-03,2026-06-29,6661



Shape precios limpios: (2097, 4)
Shape dataset mercado: (2097, 45)
Shape catálogo carteras: (286, 8)

Distribución perfiles:


perfil
conservador    154
equilibrado     97
dinamico        35
Name: count, dtype: int64


Nulos dataset mercado:


date                      0
price_XEON.DE             0
ret_1d_XEON.DE            1
ret_log_1d_XEON.DE        1
ret_21d_XEON.DE          21
ret_63d_XEON.DE          63
ret_252d_XEON.DE        252
vol_21d_XEON.DE          21
vol_63d_XEON.DE          63
vol_252d_XEON.DE        252
price_SHY                 0
ret_1d_SHY                1
ret_log_1d_SHY            1
ret_21d_SHY              21
ret_63d_SHY              63
ret_252d_SHY            252
vol_21d_SHY              21
vol_63d_SHY              63
vol_252d_SHY            252
price_AGGU.L              0
ret_1d_AGGU.L             1
ret_log_1d_AGGU.L         1
ret_21d_AGGU.L           21
ret_63d_AGGU.L           63
ret_252d_AGGU.L         252
vol_21d_AGGU.L           21
vol_63d_AGGU.L           63
vol_252d_AGGU.L         252
price_SPY                 0
ret_1d_SPY                1
ret_log_1d_SPY            1
ret_21d_SPY              21
ret_63d_SPY              63
ret_252d_SPY            252
vol_21d_SPY              21
vol_63d_SPY         

,date,price_XEON.DE,ret_1d_XEON.DE,ret_log_1d_XEON.DE,ret_21d_XEON.DE,ret_63d_XEON.DE,ret_252d_XEON.DE,vol_21d_XEON.DE,vol_63d_XEON.DE,vol_252d_XEON.DE,...,vol_63d_SPY,vol_252d_SPY,drawdown_63d_XEON.DE,drawdown_63d_SHY,drawdown_63d_AGGU.L,drawdown_63d_SPY,corr_media_63d,euribor_3m,spread,coste_financiacion
0,2017-11-21,138.143997,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.035,0.025,0.06
1,2017-11-22,138.151993,0.000058,0.000058,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.035,0.025,0.06
2,2017-11-24,138.136993,-0.000109,-0.000109,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.035,0.025,0.06
3,2017-11-27,138.126999,-0.000072,-0.000072,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.035,0.025,0.06
4,2017-11-28,138.132004,0.000036,0.000036,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.035,0.025,0.06


,id_cartera,perfil,hhi_concentracion,w_XEON.DE,w_SHY,w_AGGU.L,w_SPY,ter_cartera
0,1,dinamico,1.00,0.0,0.0,0.0,1.0,0.00090
1,2,dinamico,0.82,0.0,0.0,0.1,0.9,0.00091
2,3,dinamico,0.68,0.0,0.0,0.2,0.8,0.00092
3,4,dinamico,0.58,0.0,0.0,0.3,0.7,0.00093
4,5,dinamico,0.52,0.0,0.0,0.4,0.6,0.00094


In [8]:
# ============================================================
# 7. Exportación
# ============================================================

dataset_mercado.to_parquet("data/entradas/dataset_mercado.parquet", compression="snappy", index=False)
catalogo_carteras.to_parquet("data/entradas/catalogo_carteras.parquet", compression="snappy", index=False)

print("Archivos exportados correctamente.")

Archivos exportados correctamente.


In [9]:
# ============================================================
# 8. Dataset de colateral corregido
# ============================================================

INITIAL_COLLATERAL = 100_000


def construir_dataset_colateral(dataset_mercado, catalogo_carteras, tickers):
    mercado = dataset_mercado.copy()
    carteras = catalogo_carteras.copy()

    ret_cols = [f"ret_1d_{t}" for t in tickers]
    weight_cols = [f"w_{t}" for t in tickers]

    fechas = mercado["date"].to_numpy()
    ids = carteras["id_cartera"].to_numpy()

    R = mercado[ret_cols].fillna(0).to_numpy()
    W = carteras[weight_cols].to_numpy()

    # Retorno diario real de cada cartera
    ret_carteras = R @ W.T

    df_colateral = pd.DataFrame({
        "date": np.repeat(fechas, len(ids)),
        "id_cartera": np.tile(ids, len(fechas)),
        "ret_1d_cartera": ret_carteras.reshape(-1)
    })

    # Datos del catálogo
    df_colateral = df_colateral.merge(
        carteras,
        on="id_cartera",
        how="left"
    )

    # Variables comunes de mercado
    cols_mercado = [
        "date",
        "corr_media_63d",
        "euribor_3m",
        "spread",
        "coste_financiacion"
    ]

    df_colateral = df_colateral.merge(
        mercado[cols_mercado],
        on="date",
        how="left"
    )

    # Valor del colateral
    df_colateral["valor_colateral"] = (
        INITIAL_COLLATERAL *
        df_colateral
        .groupby("id_cartera")["ret_1d_cartera"]
        .transform(lambda x: (1 + x.fillna(0)).cumprod())
    )

    # Retornos acumulados de cartera
    df_colateral["ret_21d_cartera"] = (
        df_colateral.groupby("id_cartera")["valor_colateral"].pct_change(21, fill_method=None)
    )

    df_colateral["ret_63d_cartera"] = (
        df_colateral.groupby("id_cartera")["valor_colateral"].pct_change(63, fill_method=None)
    )

    df_colateral["ret_252d_cartera"] = (
        df_colateral.groupby("id_cartera")["valor_colateral"].pct_change(252, fill_method=None)
    )

    # Volatilidad real de cartera desde su propio retorno
    df_colateral["vol_21d_cartera"] = (
        df_colateral
        .groupby("id_cartera")["ret_1d_cartera"]
        .transform(lambda x: x.rolling(window=21, min_periods=21).std() * np.sqrt(252))
    )

    df_colateral["vol_63d_cartera"] = (
        df_colateral
        .groupby("id_cartera")["ret_1d_cartera"]
        .transform(lambda x: x.rolling(window=63, min_periods=63).std() * np.sqrt(252))
    )

    df_colateral["vol_252d_cartera"] = (
        df_colateral
        .groupby("id_cartera")["ret_1d_cartera"]
        .transform(lambda x: x.rolling(window=252, min_periods=252).std() * np.sqrt(252))
    )

    # Drawdown real de cartera
    df_colateral["max_rolling_63d_cartera"] = (
        df_colateral
        .groupby("id_cartera")["valor_colateral"]
        .transform(lambda x: x.rolling(window=63, min_periods=63).max())
    )

    df_colateral["drawdown_63d_cartera"] = (
        df_colateral["valor_colateral"] /
        df_colateral["max_rolling_63d_cartera"] - 1
    )

    df_colateral = df_colateral.drop(columns=["max_rolling_63d_cartera"])

    # VaR y Expected Shortfall históricos rolling
    df_colateral["var_63d_95"] = (
        df_colateral
        .groupby("id_cartera")["ret_1d_cartera"]
        .transform(lambda x: x.rolling(window=63, min_periods=63).quantile(0.05))
    )

    def expected_shortfall_95(x):
        q = x.quantile(0.05)
        return x[x <= q].mean()

    df_colateral["es_63d_95"] = (
        df_colateral
        .groupby("id_cartera")["ret_1d_cartera"]
        .transform(lambda x: x.rolling(window=63, min_periods=63).apply(expected_shortfall_95, raw=False))
    )

    # Rentabilidad neta anual
    df_colateral["rentabilidad_neta_252d"] = (
        df_colateral["ret_252d_cartera"]
        - df_colateral["ter_cartera"]
        - df_colateral["coste_financiacion"]
    )

    return df_colateral

In [10]:
dataset_colateral = construir_dataset_colateral(
    dataset_mercado=dataset_mercado,
    catalogo_carteras=catalogo_carteras,
    tickers=NOMBRE_ACTIVOS
)

print("Shape dataset colateral:", dataset_colateral.shape)

display(dataset_colateral.head())
display(dataset_colateral.isnull().sum())

Shape dataset colateral: (599742, 25)


,date,id_cartera,ret_1d_cartera,perfil,hhi_concentracion,w_XEON.DE,w_SHY,w_AGGU.L,w_SPY,ter_cartera,...,ret_21d_cartera,ret_63d_cartera,ret_252d_cartera,vol_21d_cartera,vol_63d_cartera,vol_252d_cartera,drawdown_63d_cartera,var_63d_95,es_63d_95,rentabilidad_neta_252d
0,2017-11-21,1,0.0,dinamico,1.00,0.0,0.0,0.0,1.0,0.00090,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2017-11-21,2,0.0,dinamico,0.82,0.0,0.0,0.1,0.9,0.00091,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2017-11-21,3,0.0,dinamico,0.68,0.0,0.0,0.2,0.8,0.00092,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2017-11-21,4,0.0,dinamico,0.58,0.0,0.0,0.3,0.7,0.00093,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2017-11-21,5,0.0,dinamico,0.52,0.0,0.0,0.4,0.6,0.00094,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


date                          0
id_cartera                    0
ret_1d_cartera                0
perfil                        0
hhi_concentracion             0
w_XEON.DE                     0
w_SHY                         0
w_AGGU.L                      0
w_SPY                         0
ter_cartera                   0
corr_media_63d            18304
euribor_3m                    0
spread                        0
coste_financiacion            0
valor_colateral               0
ret_21d_cartera            6006
ret_63d_cartera           18018
ret_252d_cartera          72072
vol_21d_cartera            5720
vol_63d_cartera           17732
vol_252d_cartera          71786
drawdown_63d_cartera      17732
var_63d_95                17732
es_63d_95                 17732
rentabilidad_neta_252d    72072
dtype: int64

In [11]:
dataset_colateral.to_parquet("data/entradas/dataset_colateral.parquet", compression="snappy", index=False)

print("Dataset colateral exportado correctamente.")

Dataset colateral exportado correctamente.


In [12]:
dataset_colateral["valor_colateral"].describe()

count    599742.000000
mean     121993.815854
std       25508.141699
min       90018.161198
25%      104932.459588
50%      113120.658533
75%      129795.372746
max      333626.800951
Name: valor_colateral, dtype: float64

In [ ]:
dataset_colateral[
    dataset_colateral["id_cartera"] == 1
][["date","ret_1d_cartera","valor_colateral"]].head(15)

,date,ret_1d_cartera,valor_colateral
0,2017-11-21,0.000000,100000.000000
286,2017-11-22,-0.000884,99911.565958
572,2017-11-24,0.002310,100142.330361
858,2017-11-27,-0.000499,100092.331295
1144,2017-11-28,0.010145,101107.733641
1430,2017-11-29,-0.000609,101046.197363
1716,2017-11-30,0.008755,101930.867035
2002,2017-12-01,-0.002076,101719.293243
2288,2017-12-04,-0.001210,101596.254283
2574,2017-12-05,-0.003597,101230.826356


In [14]:
dataset_colateral[
    [
        "var_63d_95",
        "es_63d_95"
    ]
].describe()

,var_63d_95,es_63d_95
count,582010.000000,582010.000000
mean,-0.004682,-0.006411
std,0.004606,0.006661
min,-0.056947,-0.085211
25%,-0.006026,-0.008208
50%,-0.003306,-0.004437
75%,-0.001746,-0.002328
max,0.000008,-0.000010


In [15]:
dataset_colateral.describe()

,date,id_cartera,ret_1d_cartera,hhi_concentracion,w_XEON.DE,w_SHY,w_AGGU.L,w_SPY,ter_cartera,corr_media_63d,...,ret_21d_cartera,ret_63d_cartera,ret_252d_cartera,vol_21d_cartera,vol_63d_cartera,vol_252d_cartera,drawdown_63d_cartera,var_63d_95,es_63d_95,rentabilidad_neta_252d
count,599742,599742.000000,599742.000000,599742.00000,599742.000000,599742.000000,599742.000000,599742.000000,599742.000000,581438.000000,...,593736.000000,581724.000000,527670.000000,594022.000000,582010.000000,527956.000000,582010.000000,582010.000000,582010.000000,527670.000000
mean,2022-03-11 04:06:31,143.500000,0.000208,0.46000,0.250000,0.250000,0.250000,0.250000,0.001100,0.075966,...,0.004328,0.012730,0.053535,0.047260,0.049503,0.054000,-0.010224,-0.004682,-0.006411,-0.007565
min,2017-11-21 00:00:00,1.000000,-0.109424,0.26000,0.000000,0.000000,0.000000,0.000000,0.000900,-0.122980,...,-0.327513,-0.294637,-0.186559,0.000432,0.000857,0.001003,-0.337173,-0.056947,-0.085211,-0.247459
25%,2020-01-21 00:00:00,72.000000,-0.000948,0.36000,0.100000,0.100000,0.100000,0.100000,0.001000,-0.005739,...,-0.001553,-0.000018,0.016177,0.018268,0.019588,0.022810,-0.010897,-0.006026,-0.008208,-0.044943
50%,2022-03-11 00:00:00,143.500000,0.000178,0.42000,0.200000,0.200000,0.200000,0.200000,0.001080,0.070518,...,0.003906,0.011339,0.050450,0.033448,0.036061,0.040919,-0.002930,-0.003306,-0.004437,-0.010694
75%,2024-05-02 00:00:00,215.000000,0.001496,0.54000,0.400000,0.400000,0.400000,0.400000,0.001180,0.142376,...,0.011387,0.025848,0.086426,0.059774,0.063203,0.072350,-0.000401,-0.001746,-0.002328,0.025342
max,2026-06-29 00:00:00,286.000000,0.105020,1.00000,1.000000,1.000000,1.000000,1.000000,0.001500,0.394419,...,0.251760,0.351740,0.850787,0.929686,0.599337,0.337386,0.000000,0.000008,-0.000010,0.789887
std,NaN,82.560653,0.004260,0.14697,0.229129,0.229129,0.229129,0.229129,0.000124,0.103107,...,0.018221,0.029037,0.069185,0.048924,0.047067,0.044052,0.018673,0.004606,0.006661,0.069212
